# CHARM Algorithm Demo

**CHARM** (Closed Hashing Associating Rules Mining) is an efficient algorithm for
mining all **Frequent Closed Itemsets** (FCIs) from a transaction database using a
*vertical* (tidset-based) data representation.

**Reference:** Zaki, M.J. & Hsiao, C.-J. (2002). *CHARM: An Efficient Algorithm for
Closed Itemset Mining.* SIAM International Conference on Data Mining, pp. 457–473.

---
## Outline
1. What are closed itemsets and why they matter
2. The CHARM algorithm (overview)
3. Live demo on the toy database
4. Supermarket basket example
5. Benchmark run on synthetic data


In [ ]:
# Load the algorithm (paths are relative to the notebooks/ directory)
include(joinpath("..", "src", "algorithm", "charm.jl"))
println("CHARM algorithm loaded ✓")

---
## 1. Background: Closed Itemsets

An itemset **X** is *closed* if there is no proper superset **Y ⊃ X** with the same
support count.  Closed itemsets are a *lossless* compressed representation of all
frequent itemsets: every frequent itemset and its support can be derived from the
set of FCIs alone.

| Property | All FIs | Closed FIs (FCIs) |
|----------|---------|------------------|
| Completeness | ✓ | ✓ (lossless) |
| Redundancy | high | none |
| Count | exponential | much smaller |


---
## 2. Algorithm Overview

CHARM uses a **depth-first search** over the itemset lattice combined with four
pruning properties that exploit tidset relationships:

| Property | Condition | Action |
|----------|-----------|--------|
| 1 | T(Xi) and T(Xj) incomparable | Add (Xi∪Xj, T∩) as new child |
| 2 | T(Xi) ⊆ T(Xj) | Replace Xi with Xi∪Xj (same tidset) |
| 3 | T(Xi) ⊇ T(Xj) | Add (Xi∪Xj, Tj) as child; remove Xj |
| 4 | T(Xi) = T(Xj) | Replace Xi with Xi∪Xj; remove Xj |

After recursion, each surviving node is added to the output if not already
subsumed by a previously found closed itemset.

---
## 3. Toy Example

Classic CHARM paper dataset: 10 transactions over 5 items {a, b, c, d, e}.

In [ ]:
txns_toy = [
    ["a","b","c","d","e"],
    ["a","b","c","d"],
    ["a","b","c","e"],
    ["a","b","d","e"],
    ["a","c","d","e"],
    ["b","c","d","e"],
    ["a","b","c"],
    ["a","b","d"],
    ["a","c","d"],
    ["b","c","d"],
]

println("Transaction database ($(length(txns_toy)) transactions):")
for (i, t) in enumerate(txns_toy)
    println("  T$i: {$(join(t, ", "))}")
end

In [ ]:
result_toy = charm(txns_toy, 3)
print_results(result_toy)

In [ ]:
# Compare with all-FI count (brute force)
all_items = ["a","b","c","d","e"]
all_fis = 0
for k in 1:length(all_items)
    for combo in Iterators.filter(c -> true, Iterators.map(collect, Iterators.product(fill([false,true], length(all_items))...)))
        # skip — use simpler approach below
    end
end

# Simpler: enumerate all subsets via bitmask
all_fis = 0
n = length(all_items)
for mask in 1:(2^n - 1)
    subset = [all_items[i] for i in 1:n if (mask >> (i-1)) & 1 == 1]
    sup = count(t -> all(item -> item in t, subset), txns_toy)
    if sup >= 3
        all_fis += 1
    end
end

println("All frequent itemsets (min_sup=3): $all_fis")
println("Closed frequent itemsets (CHARM): $(length(result_toy))")
println("Compression ratio: $(round(length(result_toy)/all_fis*100, digits=1))%")

---
## 4. Supermarket Basket Example

In [ ]:
path_basket = joinpath("..", "data", "toy", "example2.dat")
txns_basket = read_transactions(path_basket)

println("Basket transactions ($(length(txns_basket)) transactions):")
for (i, t) in enumerate(txns_basket)
    println("  T$i: {$(join(t, ", "))}")
end

result_basket = charm(txns_basket, 2)
println()
print_results(result_basket)

---
## 5. Benchmark Run

In [ ]:
path_bench = joinpath("..", "data", "benchmark", "T10I4D1000.dat")
txns_bench = read_transactions(path_bench)
println("Loaded $(length(txns_bench)) transactions")

for min_sup in [0.20, 0.10, 0.05]
    t0 = time()
    r = charm(txns_bench, min_sup)
    elapsed = time() - t0
    println("  min_sup=$(Int(round(min_sup*100)))%: $(length(r)) FCIs in $(round(elapsed*1000, digits=1)) ms")
end

---
## 6. Saving Results

In [ ]:
# Write results to a file
out_path = joinpath("..", "data", "toy", "example1_results.txt")
result_ex1 = charm(read_transactions(joinpath("..", "data", "toy", "example1.dat")), 3)
write_results(result_ex1, out_path)
println("Results written to $out_path")

# Preview
for line in readlines(out_path)
    println(line)
end